# 🔧 Notebook 2 — Feature Engineering
**Formula 1 ML Analytics Project**

Computes ~80 ML features in 5 categories. **No data leakage**: all rolling/cumulative
features use only past data (shift(1) before aggregation).

| Category | Features | Key Method |
|----------|----------|-----------|
| A — Driver Skill | 14 | cumsum + shift(1), rolling(5/10) |
| B — Constructor | 7 | rolling team aggregates |
| C — Circuit | 7 | expanding historical averages |
| D — Race Context | 6 | per-race transforms |
| E — Era/Teammate | 5 | self-join, season normalization |


In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, "..")
DATA_PATH = "../data/processed/"

# Load master_df (output from notebook 1)
master_file = os.path.join(DATA_PATH, "master_df.csv")
if not os.path.exists(master_file):
    print("⚠️  master_df.csv not found. Run notebook 01_data_prep.ipynb first.")
    print(f"Expected: {os.path.abspath(master_file)}")
else:
    master_df = pd.read_csv(master_file, low_memory=False)
    print(f"✅ Loaded master_df: {master_df.shape}")
    master_df.head(2)


In [ ]:
from src.feature_engineering import (
    add_driver_skill_features,
    add_constructor_features,
    add_circuit_features,
    add_race_context_features,
    add_normalization_features,
    engineer_features,
    get_feature_columns,
    get_feature_groups
)

# Run full feature engineering pipeline
print("Running feature engineering pipeline...")
print("(This may take 1-3 minutes for the full 1950-2024 dataset)")
featured_df = engineer_features(master_df)
print(f"\n✅ Feature engineering complete!")
print(f"Shape: {featured_df.shape}")
print(f"New feature columns: {len(get_feature_columns())}")


In [ ]:
# Review feature groups
feature_groups = get_feature_groups()
print("Feature groups:")
for group, cols in feature_groups.items():
    print(f"\n  {group} ({len(cols)} features):")
    for c in cols:
        print(f"    - {c}")


In [ ]:
# Check for data leakage — verify rolling features are shifted
print("=== DATA LEAKAGE CHECK ===")
print("Verifying that rolling features don't include current race...")

# Pick a sample driver and check that career_wins_so_far < actual career wins
if 'career_wins_so_far' in featured_df.columns and 'driver_name' in featured_df.columns:
    sample = featured_df[featured_df['driver_name'].str.contains('Hamilton', na=False)].sort_values(['year','round'])
    if len(sample) > 0:
        print(f"\nLewis Hamilton — first 5 races:")
        check_cols = ['year', 'round', 'positionOrder', 'career_wins_so_far', 'career_races_so_far']
        available = [c for c in check_cols if c in sample.columns]
        print(sample[available].head(5).to_string())
        print("\n→ career_wins_so_far at race N should equal total wins BEFORE race N ✓")


In [ ]:
# Feature correlation with race position
feature_cols = get_feature_columns()
available_features = [c for c in feature_cols if c in featured_df.columns]
target = 'positionOrder'

if target in featured_df.columns and available_features:
    corr = featured_df[available_features + [target]].corr()[target].drop(target)
    corr_sorted = corr.abs().sort_values(ascending=False)
    
    print(f"Top 15 features correlated with race position:")
    print(corr_sorted.head(15).to_string())
    
    plt.figure(figsize=(10, 7))
    colors = ['#DC0000' if c > 0 else '#0600EF' for c in corr_sorted.head(20)]
    corr_sorted.head(20).plot(kind='barh', color=colors, figsize=(10,7))
    plt.xlabel('|Pearson r| with positionOrder')
    plt.title('Feature Correlation with Race Finish Position')
    plt.tight_layout()
    plt.show()


In [ ]:
# Save featured DataFrame
OUTPUT_PATH = "../data/processed/"
os.makedirs(OUTPUT_PATH, exist_ok=True)
out_file = os.path.join(OUTPUT_PATH, "featured_df.csv")
featured_df.to_csv(out_file, index=False)
print(f"✅ Saved featured_df.csv → {out_file}")
print(f"   Shape: {featured_df.shape}")
print(f"   Columns: {list(featured_df.columns)[:10]} ... ({len(featured_df.columns)} total)")
